# transform

> `Transform` and `Pipeline` — the Julia analog of fasttransform's
> `transform.py`. Ported from the real source (see
> `fasttransform_files/ARCHITECTURE.md`), not just the README.
>
> The headline simplification: everything Python's `_TfmMeta`/`_TfmDict`
> metaclass machinery and the third-party `plum` dispatch library exist
> for — "define several functions with the same name, different
> parameter type annotations, resolved by the argument's runtime type" —
> is exactly what Julia's own native multiple dispatch already does, for
> free. A `Transform` subtype just defines ordinary `encodes`/`decodes`
> methods with whatever type annotations it needs; Julia's method table
> handles the rest, including the "no match → fall through to a default"
> behavior, via one blanket fallback method on the abstract type — no
> metaclass, no dict interception, no merging-methods-across-inheritance
> logic.

In [ ]:
#| default_exp transform

In [ ]:
#| hide
using FTransform

## `Transform`

`encodes`/`decodes`/`setups!` all have a blanket passthrough/no-op default
on the abstract type — this *is* the "no dispatch match → do nothing"
behavior, for free, since Julia always finds at least this fallback
method applicable. `order` controls `Pipeline` sequencing (lower runs
first). `autoretain` is the one real, documented fidelity gap from the
Python original: fasttransform lets a *specific encodes/decodes call* opt
out of type-preservation via a `-> None` return annotation; Julia's own
`::T` return-type annotation already does something different (an
automatic runtime convert, not an inspectable hint), so it can't double
as that same opt-out signal without conflicting with it. `autoretain` is
a coarser, per-Transform-*type* opt-out instead of per-method — override
it to `false` on a Transform subtype that wants its `encodes`/`decodes`
results returned exactly as computed, never auto-preserved.

In [ ]:
#| export
"A reusable, optionally-reversible data transformation. Subtype this and define `encodes`/`decodes`/`setups!` methods (with whatever argument types you need — Julia's own dispatch picks the right one)."
abstract type Transform end

"encodes(t, x): apply `t`'s forward transform to `x`. Default: passthrough (no method defined for `x`'s type)."
encodes(t::Transform, x) = x

"decodes(t, x): apply `t`'s inverse transform to `x`, undoing [`encodes`](#encodes). Default: passthrough."
decodes(t::Transform, x) = x

"setups!(t, items): configure `t` from a dataset `items` (e.g. computing a mean/std to normalize by). Default: no-op."
setups!(t::Transform, items) = nothing

"Priority for `Pipeline` ordering — lower runs first. Default `0`."
order(t::Transform) = 0

"Does `t`'s call wrapper auto-preserve/restore the input's runtime type on the result (see the module docstring)? Default `true`."
autoretain(t::Transform) = true

## Calling a `Transform`

A tuple argument is unpacked and each element transformed independently
(then the result is re-packed as a tuple via [`retain_type`](#retain_type) —
matching fasttransform's own per-element tuple handling).

In [ ]:
#| export
_is_tuple(x) = x isa Tuple

function _apply(f::Function, t::Transform, x)
    if _is_tuple(x)
        y = Tuple(_apply(f, t, xi) for xi in x)
        return retain_type(y, x)
    end
    y = f(t, x)
    return autoretain(t) ? retain_type(y, x) : y
end

"Calling a `Transform` applies [`encodes`](#encodes)."
(t::Transform)(x) = _apply(encodes, t, x)

"    decode(t, x)

Apply [`decodes`](#decodes) to `x`, undoing `t`'s forward transform."
decode(t::Transform, x) = _apply(decodes, t, x)

"""
    setup!(t, items=nothing)

Call [`setups!`](#setups!) to configure `t` from `items`, returning `t`.
"""
function setup!(t::Transform, items = nothing)
    setups!(t, items)
    return t
end

## Wrapping a plain function

`transform(fn)` (and `transform(fn, dec)` for a reversible one) wraps an
ordinary function as a `Transform`, matching fasttransform's
`Transform(fn)` constructor usage — the quick-start's `@Transform def
add_one(x): ...` decorator has no direct Julia equivalent (no
decorators), so this is a plain function call instead: `transform(add_one)`.

For the "pass several functions with different type annotations" case
(fasttransform's `Transform(enc=(inc1,inc2))`), Julia doesn't need a
special multi-function constructor at all — just define `inc` as one
*named* function with multiple dispatch methods and pass that single
function in; its methods already do the dispatching.

In [ ]:
#| export
"A `Transform` built from a plain function (or a pair, for a reversible one), rather than a `Transform` subtype's own methods."
struct FuncTransform <: Transform
    enc::Union{Nothing,Function}
    dec::Union{Nothing,Function}
    ord::Int
    retain::Bool
end

"""
    transform(enc=nothing; dec=nothing, order=0, autoretain=true)

Wrap plain function(s) as a [`Transform`](#Transform). `enc`/`dec` can be
`nothing` (passthrough) or any callable — including an already-`Transform`
wrapped one, in which case it's used as-is (matching fasttransform's own
"already a Transform? leave it alone" rule in `mk_transform`).
"""
function transform(enc = nothing; dec = nothing, order::Int = 0, autoretain::Bool = true)
    FuncTransform(enc, dec, order, autoretain)
end

encodes(t::FuncTransform, x) = (t.enc === nothing || !applicable(t.enc, x)) ? x : t.enc(x)
decodes(t::FuncTransform, x) = (t.dec === nothing || !applicable(t.dec, x)) ? x : t.dec(x)
order(t::FuncTransform) = t.ord
autoretain(t::FuncTransform) = t.retain

"Already a `Transform`? Use it as-is. A plain function? Wrap it with [`transform`](#transform)."
mk_transform(t::Transform) = t
mk_transform(f) = transform(f)

## `Pipeline`

An ordered sequence of `Transform`s, sorted by [`order`](#order).
`encode`/calling applies them forward; `decode` undoes them in *reverse*
order — the same principle as inverting a composed function,
`(f∘g)⁻¹ = g⁻¹∘f⁻¹`.

In [ ]:
#| export
"A sequence of `Transform`s, applied in order (and undone in reverse order by [`decode`](#decode))."
struct Pipeline
    fs::Vector{Transform}
end

"""
    Pipeline(fs=Transform[])

Build a `Pipeline` from a collection of transforms (plain functions are
wrapped via [`mk_transform`](#mk_transform)), sorted by [`order`](#order).
"""
function Pipeline(fs::AbstractVector = Transform[])
    fs isa Pipeline && return fs
    ts = Transform[mk_transform(f) for f in fs]
    sort!(ts; by = order)
    return Pipeline(ts)
end

(p::Pipeline)(x) = foldl((acc, t) -> t(acc), p.fs; init = x)
decode(p::Pipeline, x) = foldl((acc, t) -> decode(t, acc), reverse(p.fs); init = x)

"Run [`setup!`](#setup!) on every transform in `p`, in order, each from the same `items`."
function setup!(p::Pipeline, items = nothing)
    for t in p.fs
        setup!(t, items)
    end
    return p
end

"""
    add!(p, t, items=nothing)

Add transform `t` to `p` (after calling `t`'s own [`setup!`](#setup!) on
`items`), re-sorting by [`order`](#order).
"""
function add!(p::Pipeline, t, items = nothing)
    tfm = mk_transform(t)
    setup!(tfm, items)
    push!(p.fs, tfm)
    sort!(p.fs; by = order)
    return p
end

Base.length(p::Pipeline) = length(p.fs)
Base.getindex(p::Pipeline, i) = p.fs[i]

## Tests

In [ ]:
#| hide
# README quick-start examples, ported 1:1 as acceptance tests.

# Transform from a plain function
add_one(x::Int) = x + 1
t1 = transform(add_one)
@assert t1(2) == 3
@assert t1(2.0) == 2.0  # no matching method for Float64 -> passthrough, unlike Python no error either

# Reversibility
enc(x) = x * 2
dec(x) = x ÷ 2
t2 = transform(enc; dec = dec)
@assert t2(2) == 4
@assert decode(t2, 2) == 1
@assert decode(t2, t2(2)) == 2

# Setup: z-score normalization storing mean/std from a dataset
mutable struct NormalizeMean <: Transform
    mean::Float64
    std::Float64
    NormalizeMean() = new(0.0, 1.0)
end
function setups!(t::NormalizeMean, items)
    t.mean = sum(items) / length(items)
    t.std = sqrt(sum((x - t.mean)^2 for x in items) / (length(items) - 1))
end
encodes(t::NormalizeMean, x) = (x - t.mean) / t.std
decodes(t::NormalizeMean, x) = x * t.std + t.mean
normalize = NormalizeMean()
setup!(normalize, [1, 2, 3, 4, 5])
@assert normalize.mean == 3.0

# Type-based multiple dispatch — one named Julia function, two methods,
# no special multi-function constructor needed (unlike fasttransform's
# `Transform(enc=(inc1,inc2))`).
incdisp(x::Int) = x + 1
incdisp(x::AbstractString) = x * "a"
t3 = transform(incdisp)
@assert t3(5) == 6
@assert t3("b") == "ba"

# Type conversion and preservation, via the FS wrapper type from 00_cast.ipynb's tests
struct FS2 <: Real
    val::Float64
end
Base.Float64(x::FS2) = x.val
FS2(x::Real) = FS2(Float64(x))
Base.:*(a::FS2, b::Real) = Float64(a) * b
double(x) = x * 2.0  # no explicit return type -> preserve caller's wrapper type
t4 = transform(double)
fs1 = FS2(1.0)
val = t4(fs1)
@assert val isa FS2
@assert Float64(val) == 2.0

# Pipeline: compose, then decode reverses in the opposite order
halve(x) = x / 2.0
dt = transform(double; dec = halve)
p = Pipeline([dt, normalize])
v = p(5)
@assert decode(p, v) == 5.0
@assert length(p) == 2
